# ASSIGNMENT 6


## 1. Design Backed support for the HR Access-Control Problem

A company's HR system models employees using a class hierarchy rooted at Employee, with Manager as a specialization. The system has three data-protection requirements:
- The **employee's name** must be freely readable and writable from anywhere in the codebase.
- The **salary** must never be overwritten directly by assigning to it from outside the class. It may only change through a method that revises it by a **percentage hike** between 0 and 50 inclusive; any other value must raise an Error.
- The **leave_balance** must be usable by Employee itself and by any class that extends it, such as Manager, but code outside this class family should not be able to rely on accessing it directly.
- Implement Employee and a Manager subclass with apply_leave(days), which reduces leave_balance and raises an Error if days exceed the remaining balance.

Tasks:
- Implement the Employee class using appropriate access-control conventions.
- Implement the Manager subclass.
- Implement salary revision using a percentage hike between 0 and 50 inclusive.
- Implement apply_leave(days).
- Demonstrate that attaching '__salary' to an Employee instance from outside the class does not read the real salary. Explain what Python actually did to the attribute name.
- Create an employee/manager object and perform some salary and leave operations.
- **File Handling:** Create a text file named employee_records.txt.
- Store the following information in the file:
    - Employee name
    - Salary
    - Remaining leave balance
- Store the information in the file.
- Read the stored information back and display it.
- Properly close the file.

[ Encapsulation+ Text File Handling open(), w, write(), read(), close() ]


In [ ]:
class Employee:
    def __init__(self, name, salary, leave_balance):
        self.name = name
        self.__salary = salary
        self._leave_balance = leave_balance
        
    def revise_salary(self, percentage_hike):
        if 0 <= percentage_hike <= 50:
            self.__salary += self.__salary * (percentage_hike / 100)
        else:
            raise ValueError("Percentage hike must be between 0 and 50 inclusive.")
            
    def get_salary(self):
        return self.__salary
        
    def apply_leave(self, days):
        if days <= self._leave_balance:
            self._leave_balance -= days
            print(f"Leave applied for {days} days. Remaining balance: {self._leave_balance}")
        else:
            raise ValueError("Days exceed remaining leave balance.")

class Manager(Employee):
    pass 

emp = Employee("Alice", 50000, 20)
mgr = Manager("Bob", 80000, 25)

emp.revise_salary(10)
mgr.apply_leave(5)

emp.__salary = 100000
print("Attached __salary from outside:", emp.__salary)
print("Actual salary via getter:", emp.get_salary())

# Explanation: Python uses name mangling for attributes starting with double underscores (__).
# The real salary is stored internally as _Employee__salary. Attaching __salary from outside just creates a new normal attribute named __salary, without overwriting the actual salary data.

with open("employee_records.txt", "w") as f:
    f.write(f"Name: {emp.name}, Salary: {emp.get_salary()}, Remaining leave balance: {emp._leave_balance}\n")
    f.write(f"Name: {mgr.name}, Salary: {mgr.get_salary()}, Remaining leave balance: {mgr._leave_balance}\n")

with open("employee_records.txt", "r") as f:
    print("\n--- File Content ---")
    print(f.read())


## 2. Code in Python for an e-commerce checkout system needs a common contract for payment gateways (Credit Card, UPI, and others added later). The design must make it structurally impossible to create a generic, gateway-less payment object — every concrete gateway must be forced to supply its own payment logic, while still inheriting a shared receipt-formatting method from the common ancestor.

Tasks:
- Design the **base class** and **two concrete gateways**, [CreditCardProcessor and UPIProcessor] that satisfy the requirement above.
- Show that attempting to instantiate the base class directly fails, and capture/print the exact exception message.
- Process a payment of 1500 through both concrete gateways and print their receipts.
- **File Handling:** Create a text file named payment_receipts.txt.
- Store the receipts generated by both payment processors in the file.
- Use write() to store each receipt.
- Read the file using readlines() and display all stored receipts.

[Abstraction + Text File Handling - w, write(), readlines(), close() ]


In [ ]:
from abc import ABC, abstractmethod

class PaymentGateway(ABC):
    @abstractmethod
    def process_payment(self, amount):
        pass
        
    def format_receipt(self, amount, gateway_name):
        return f"Receipt: Payment of {amount} processed successfully via {gateway_name}."

class CreditCardProcessor(PaymentGateway):
    def process_payment(self, amount):
        return self.format_receipt(amount, "Credit Card")

class UPIProcessor(PaymentGateway):
    def process_payment(self, amount):
        return self.format_receipt(amount, "UPI")

try:
    pg = PaymentGateway()
except TypeError as e:
    print("Failed to instantiate base class:", e)

cc = CreditCardProcessor()
upi = UPIProcessor()

r1 = cc.process_payment(1500)
r2 = upi.process_payment(1500)
print(r1)
print(r2)

with open("payment_receipts.txt", "w") as f:
    f.write(r1 + "\n")
    f.write(r2 + "\n")

with open("payment_receipts.txt", "r") as f:
    print("\n--- File Content ---")
    receipts = f.readlines()
    for r in receipts:
        print(r.strip())


## 3. WAP for A notification service must send the same logical message through Email, SMS, and Push channels, but each channel formats the message differently: Email tags it, SMS truncates the message to its first 20 characters, and Push uppercases it. Calling code should never need to know which concrete channel it is talking to.

Tasks:
- Design a common Notifier ancestor whose send(message) is meant to be replaced by every channel.
- Make it fail loudly if a subclass forgets to supply its own version.
- Implement:
    - EmailNotifier
    - SMSNotifier
    - PushNotifier
- Implement the required formatting rules.
- Write a **broadcast(notifiers, message)** function that sends one message through a heterogeneous list using a single loop.
- Run it using: 'Server maintenance scheduled tonight'
- Display the notification generated by each channel.
- **File Handling:** Create a file named notifications.txt.
- Store every generated notification in the file.
- Use **append mode (a)** so that new notification messages are added without deleting previously stored notifications.
- Use write() to append each notification.
- Read the complete file and display all stored notifications.

[ Method Overriding + File Handling concepts: a, write(), read(), close() ]


In [ ]:
class Notifier:
    def send(self, message):
        raise NotImplementedError("Subclasses must implement send()")

class EmailNotifier(Notifier):
    def send(self, message):
        return f"[Email] {message}"

class SMSNotifier(Notifier):
    def send(self, message):
        return message[:20]

class PushNotifier(Notifier):
    def send(self, message):
        return message.upper()

def broadcast(notifiers, message):
    results = []
    for notifier in notifiers:
        results.append(notifier.send(message))
    return results

notifiers = [EmailNotifier(), SMSNotifier(), PushNotifier()]
msg = "Server maintenance scheduled tonight"
results = broadcast(notifiers, msg)

for r in results:
    print(r)

with open("notifications.txt", "a") as f:
    for r in results:
        f.write(r + "\n")

with open("notifications.txt", "r") as f:
    print("\n--- File Content ---")
    print(f.read())


## 4. WAP for the Calculator class needs a single method named add whose behavior depends on the type of arguments provided. The method should accept two or more arguments:
For **integers**, all arguments should be summed arithmetically.
For **strings**, all arguments should be joined using a single space, with Unnecessary leading and trailing spaces removed.
For **lists**, all lists should be merged into a single sorted list containing only unique elements.
**Passing mismatched types**, such as an integer and a string, should fail clearly.

Tasks:
- If you simply write def add(self, a, b): multiple times with different bodies in the same class, what actually happens in Python? State the rule and its consequence in 1–2 sentences.
- Implement the Calculator class with a single add method that supports two or more arguments and satisfies all three behaviors described above. Use an appropriate mechanism in Python to implement method overloading-like behavior based on argument types.
- **Test the method with:**
    - add(3, 4)
    - add(3, 4, 5, 6)
    - add("hello", "world")
    - add("hello", "world", "python")
    - add([1, 2, 3], [2, 3, 4])
    - add([1, 2], [2, 3], [4, 5])
- Test the method with **mismatched types**: add(3, "4")
- The program should produce a clear Validation.
- State one limitation of this approach compared with true compile-time method overloading in languages such as Java or C++.

[Method overloading ]


In [ ]:
class Calculator:
    def add(self, *args):
        if not args:
            return None
        
        first_type = type(args[0])
        for arg in args:
            if type(arg) != first_type:
                raise TypeError("Mismatched types provided.")
                
        if first_type == int:
            return sum(args)
        elif first_type == str:
            return " ".join([arg.strip() for arg in args])
        elif first_type == list:
            merged = []
            for lst in args:
                merged.extend(lst)
            return sorted(list(set(merged)))
        else:
            raise TypeError("Unsupported type.")

calc = Calculator()
print(calc.add(3, 4))
print(calc.add(3, 4, 5, 6))
print(calc.add("hello", "world"))
print(calc.add("hello", "world", "python"))
print(calc.add([1, 2, 3], [2, 3, 4]))
print(calc.add([1, 2], [2, 3], [4, 5]))

try:
    calc.add(3, "4")
except TypeError as e:
    print("Validation Error:", e)

# 1. If you write `def add(self, a, b):` multiple times in the same class, Python will only keep the last definition and overwrite the previous ones. The consequence is that you can't have multiple methods with the same name based on different parameters in Python natively.
# 2. Limitation of this approach compared with true compile-time method overloading in languages like Java or C++ is that type checking only happens at runtime, meaning errors due to mismatched types are not caught until the code is actually executed.



## 5. A Distance class is required to represent a distance in meters. Create two or more Distance objects and provide support for performing arithmetic and comparison operations directly between objects.

The class should:
- Store the distance value in meters.
- Overload the + operator so that two or more Distance objects can be added.
- Overload the - operator to find the difference between two Distance objects.
- Overload the == operator to check whether two Distance objects represent the same distance.
- Overload the < operator to compare two Distance objects.
- Display the distance in a readable format when the object is printed.

Tasks:
- Create a Distance class with a constructor that accepts distance in meters.
    - Overload the + operator using `__add__()` so that: d1 + d2 returns a new Distance object containing the sum.
    - Extend the + operator so that multiple objects can be added: d1 + d2 + d3
    - Overload the - operator using `__sub__()`.
    - Overload the == operator using `__eq__()`.
    - Overload the < operator using `__lt__()`.
    - Overload `__str__()` so that printing an object displays the distance in meters.
- Test the class using at least three objects.
- Explain how Python translates an expression such as d1 + d2 into a special method call.
- Create a binary file named:distances.dat
- Store the Distance objects in the binary file using the pickle module.
- Write the objects.
- Read the objects back.
- Display the distances read from the binary file.
- Store at least three Distance objects and retrieve them from the file.

[operator overloading +File Handling concepts: Binary file, wb, rb, pickle.dump(), pickle.load() ]


In [ ]:
import pickle

class Distance:
    def __init__(self, meters):
        self.meters = meters
        
    def __add__(self, other):
        return Distance(self.meters + other.meters)
        
    def __sub__(self, other):
        return Distance(abs(self.meters - other.meters))
        
    def __eq__(self, other):
        return self.meters == other.meters
        
    def __lt__(self, other):
        return self.meters < other.meters
        
    def __str__(self):
        return f"{self.meters} meters"

d1 = Distance(10)
d2 = Distance(20)
d3 = Distance(30)

print(f"d1 + d2 = {d1 + d2}")
print(f"d1 + d2 + d3 = {d1 + d2 + d3}")
print(f"d2 - d1 = {d2 - d1}")
print(f"d1 == d2: {d1 == d2}")
print(f"d1 < d2: {d1 < d2}")

# Explanation: Python translates the expression `d1 + d2` into the special method call `d1.__add__(d2)`.

distances_to_store = [d1, d2, d3]
with open("distances.dat", "wb") as f:
    pickle.dump(distances_to_store, f)

with open("distances.dat", "rb") as f:
    loaded_distances = pickle.load(f)

print("\n--- Read from Binary File ---")
for d in loaded_distances:
    print(d)



## 6. Problem Statement: A media player needs to play items from AudioTrack, VideoClip, and Podcast — three classes that share no common ancestor and were written by different teams. A Playlist should be able to play any object that exposes a callable play() method, and should skip (not crash on) anything that doesn't, printing a message instead.

Tasks:
- Implement **AudioTrack, VideoClip, and Podcast**, each with its own **play()** returning a distinct message, and confirm none of them share a base class other than Object.
- Implement **Playlist.play_all()** that works correctly across all three types plus a plain string placed in the same list: not_playable. Skip the string instead of crashing.
- Print an appropriate message when an object does not provide a callable play() method.
- **Run play_all() on:**
    [
        AudioTrack(...),
        VideoClip(...),
        Podcast(...),
        "not_playable"
    ]
- Create a CSV file named: **playlist_history.csv**
- Store the result of each playlist item in the CSV file.
- Use csv.writer() to create the writer.
- Use writerow() to write each result.
- Include suitable column headings such as: Type, Status, Message
- Run the playlist again and store all results in the CSV file.
- Read the CSV file.
- Display the stored records.

[duck Typing +File Handling concepts: csv, csv.writer(), writerow(), reader(). ]


In [ ]:
import csv

class AudioTrack:
    def play(self):
        return "Playing Audio Track"

class VideoClip:
    def play(self):
        return "Playing Video Clip"

class Podcast:
    def play(self):
        return "Playing Podcast"

class Playlist:
    def play_all(self, items):
        results = []
        for item in items:
            if hasattr(item, 'play') and callable(item.play):
                message = item.play()
                print(message)
                results.append((type(item).__name__, "Success", message))
            else:
                print(f"Cannot play item of type {type(item).__name__}")
                results.append((type(item).__name__, "Skipped", "not_playable"))
        return results

playlist = Playlist()
items = [AudioTrack(), VideoClip(), Podcast(), "not_playable"]

results = playlist.play_all(items)

with open("playlist_history.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Type", "Status", "Message"])
    for r in results:
        writer.writerow(r)

with open("playlist_history.csv", "r") as f:
    reader = csv.reader(f)
    print("\n--- CSV Content ---")
    for row in reader:
        print(row)

